In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = torch.tensor([[2, 1000], [3, 2000], [2, 500], [1, 800], [4, 3000]], dtype=torch.float, device=device)
labels = torch.tensor([[19], [31], [14], [15], [43]], dtype=torch.float, device=device)


w = torch.ones(2, 1, requires_grad=True, device=device)
b = torch.ones(1, requires_grad=True, device=device)

epoch = 200
lr = 0.0000001

for i in range(epoch):
    outputs = inputs @ w + b
    loss = torch.mean(torch.square(outputs - labels))
    print("loss", loss.item())
    loss.backward()
    print("w.grad", w.grad.tolist())
    with torch.no_grad():
        w -= w.grad * lr
        b -= b.grad * lr

    w.grad.zero_()
    b.grad.zero_()

loss 2898583.75
w.grad [[8600.0], [5876040.0]]
loss 474035.15625
w.grad [[3476.080322265625], [2376262.5]]
loss 77528.75
w.grad [[1403.973876953125], [960957.0]]
loss 12684.8251953125
w.grad [[566.0172729492188], [388609.625]]
loss 2080.366943359375
w.grad [[227.14883422851562], [157153.1875]]
loss 346.13250732421875
w.grad [[90.11088562011719], [63552.50390625]]
loss 62.519161224365234
w.grad [[34.69296646118164], [25700.54296875]]
loss 16.13753318786621
w.grad [[12.282038688659668], [10393.2646484375]]
loss 8.55237102508545
w.grad [[3.2191028594970703], [4203.03076171875]]
loss 7.311903476715088
w.grad [[-0.44594287872314453], [1699.704345703125]]
loss 7.109036922454834
w.grad [[-1.9280805587768555], [687.363037109375]]
loss 7.075860500335693
w.grad [[-2.5274596214294434], [277.970947265625]]
loss 7.070433139801025
w.grad [[-2.7698497772216797], [112.4111328125]]
loss 7.069541931152344
w.grad [[-2.867867946624756], [45.4619140625]]
loss 7.06939697265625
w.grad [[-2.9074978828430176],

通过上边的代码，可以训练一个线性回归模型，你可以尝试调整学习率lr，你会发现这个lr必须设置的很小。如果设置稍大，模型训练过程就会不收敛，loss值会快速增大，直到超过float的表示范围。而且loss值降到7左右，就很难再下降了。我们造的数据是严格按照线性方程构造的，理论上loss应该可以降到非常接近0的。但为什么loss值不能下降到0呢？ 我们仔细观察第一次迭代的打印值：loss 2898583.75
w.grad [[8600.0], [5876040.0]]

根本原因就是对
w
0
 和
w
1
的训练，它们共用学习率，但是因为它们对应feature的取值范围不同，导致他们对loss函数的影响不同，进而导致它们对loss的梯度的取值范围不同。

如果我们让所有feature的取值范围相同，这样所有训练参数对loss函数的影响就相同了，计算得到的梯度都差不多，就可以用统一的学习率来进行调整了。 对于bias而言，它的系数为1，相当于它的输入feature大小永远都是1。那么我们就把其他feature都调整到1左右。最简单的做法，就是让输入feature都除以这个feature的最大值，这样所有feature的取值都是0到1之间了。 我们试一下这样是否可以改进训练的稳定性：

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = torch.tensor([[2, 1000], [3, 2000], [2, 500], [1, 800], [4, 3000]], dtype=torch.float, device=device)
labels = torch.tensor([[19], [31], [14], [15], [43]], dtype=torch.float, device=device)

#进行归一化
inputs = inputs / torch.tensor([4, 3000], device=device)


w = torch.ones(2, 1, requires_grad=True, device=device)
b = torch.ones(1, requires_grad=True, device=device)

epoch = 1000
lr = 0.5

for i in range(epoch):
    outputs = inputs @ w + b
    loss = torch.mean(torch.square(outputs - labels))
    print("loss", loss.item())
    loss.backward()
    print("w.grad", w.grad.tolist())
    with torch.no_grad():
        w -= w.grad * lr
        b -= b.grad * lr

    w.grad.zero_()
    b.grad.zero_()

loss 609.12255859375
w.grad [[-31.823333740234375], [-28.17155647277832]]
loss 276.1470947265625
w.grad [[18.713247299194336], [14.430887222290039]]
loss 130.7478485107422
w.grad [[-14.165596008300781], [-13.107965469360352]]
loss 66.27214050292969
w.grad [[7.567931175231934], [5.2582902908325195]]
loss 36.88682556152344
w.grad [[-6.486094951629639], [-6.472206115722656]]
loss 22.8634090423584
w.grad [[2.8819406032562256], [1.4812228679656982]]
loss 15.683402061462402
w.grad [[-3.105940341949463], [-3.4828484058380127]]
loss 11.647061347961426
w.grad [[0.949533224105835], [-0.009576082229614258]]
loss 9.129942893981934
w.grad [[-1.5856672525405884], [-2.083230972290039]]
loss 7.404510498046875
w.grad [[0.18418240547180176], [-0.5427929162979126]]
loss 6.133534908294678
w.grad [[-0.8760228753089905], [-1.3865240812301636]]
loss 5.151700973510742
w.grad [[-0.09213244915008545], [-0.6842262744903564]]
loss 4.371189117431641
w.grad [[-0.5246062278747559], [-1.0085372924804688]]
loss 3.7404

方法二：
实际上在深度学习里，更常用的是对特征进行标准化处理。也就是对每个feature减去自己的均值，再除以自己的标准差。这样就把这个feature转化为均值为0，标准差为1的分布了。在归一化操作里，是对每个feature除以这个feature所有样本中绝对值最大的值。只有这一个值决定缩放大小。但这个值有可能是个异常值。相比之下标准化处理会考虑所有样本的分布情况，避免缩放受异常值的影响，训练起来会更稳定。

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = torch.tensor([[2, 1000], [3, 2000], [2, 500], [1, 800], [4, 3000]], dtype=torch.float, device=device)
labels = torch.tensor([[19], [31], [14], [15], [43]], dtype=torch.float, device=device)

#计算特征的均值和标准差
mean = inputs.mean(dim=0)
std = inputs.std(dim=0)
#对特征进行标准化
inputs_norm = (inputs-mean)/std

w = torch.ones(2, 1, requires_grad=True, device=device)
b = torch.ones(1, requires_grad=True, device=device)

epoch = 1000
lr = 0.5

for i in range(epoch):
    outputs = inputs_norm @ w + b
    loss = torch.mean(torch.square(outputs - labels))
    print("loss", loss.item())
    loss.backward()
    print("w.grad", w.grad.tolist())
    with torch.no_grad():
        w -= w.grad * lr
        b -= b.grad * lr

    w.grad.zero_()
    b.grad.zero_()

loss 635.2097778320312
w.grad [[-15.604001998901367], [-16.726499557495117]]
loss 25.922626495361328
w.grad [[9.087747573852539], [8.043952941894531]]
loss 8.413028717041016
w.grad [[-4.053674697875977], [-5.024292945861816]]
loss 3.343099594116211
w.grad [[2.856461524963379], [1.953890323638916]]
loss 1.786826729774475
w.grad [[-0.8548375368118286], [-1.6941328048706055]]
loss 1.2350934743881226
w.grad [[1.0655667781829834], [0.2851123511791229]]
loss 0.980950653553009
w.grad [[0.005011647939682007], [-0.720727801322937]]
loss 0.8237902522087097
w.grad [[0.5270583629608154], [-0.14780156314373016]]
loss 0.7054709792137146
w.grad [[0.21328718960285187], [-0.4142606258392334]]
loss 0.6080939769744873
w.grad [[0.3450268507003784], [-0.2385251522064209]]
loss 0.5252779126167297
w.grad [[0.2431029975414276], [-0.2995377480983734]]
loss 0.4540571868419647
w.grad [[0.26724934577941895], [-0.23734888434410095]]
loss 0.39258116483688354
w.grad [[0.2266899198293686], [-0.24253252148628235]]
los

有一点要特别注意，假如你在训练时对数据做了归一化，那么你一定要记录你做归一化时的参数。在对数据进行预测时，首先需要先对feature用同样的参数进行归一化，然后再带入模型，得到预测值。

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = torch.tensor([[2, 1000], [3, 2000], [2, 500], [1, 800], [4, 3000]], dtype=torch.float, device=device)
labels = torch.tensor([[19], [31], [14], [15], [43]], dtype=torch.float, device=device)#目标时间，即标签

#计算每个特征的均值和标准差
mean = inputs.mean(dim=0)
std = inputs.std(dim=0)
#对特征进行标准化
inputs = (inputs-mean)/std

w = torch.ones(2, 1, requires_grad=True, device=device)
b = torch.ones(1, requires_grad=True, device=device)

epoch = 2000
lr = 0.1

for i in range(epoch):
    outputs = inputs @ w + b
    loss = torch.mean(torch.square(outputs - labels))#残差项平方均值
    print("loss", loss.item())
    loss.backward()
    print("w.grad", w.grad.tolist())
    with torch.no_grad():
        w -= w.grad * lr
        b -= b.grad * lr

    w.grad.zero_()
    b.grad.zero_()

# 对新采集的数据[3, 2500]进行预测
new_input = torch.tensor([[3,2500]],dtype=torch.float,device=device)
# 对于新的数据进行预测时，同样要进行标准化
new_input = (new_input-mean)/std
# 预测
predict = new_input @ w + b
# 打印预测结果
print("Predict:",predict.tolist()[0][0])

loss 635.2097778320312
w.grad [[-15.604001998901367], [-16.726499557495117]]
loss 393.7582092285156
w.grad [[-10.665653228759766], [-11.772409439086914]]
loss 246.2174530029297
w.grad [[-7.240629196166992], [-8.331866264343262]]
loss 155.14720153808594
w.grad [[-4.8658552169799805], [-5.941790580749512]]
loss 98.4687271118164
w.grad [[-3.2199454307556152], [-4.280794143676758]]
loss 62.9586296081543
w.grad [[-2.079848289489746], [-3.125821828842163]]
loss 40.59088897705078
w.grad [[-1.290771484375], [-2.3220789432525635]]
loss 26.439193725585938
w.grad [[-0.7452719211578369], [-1.7621188163757324]]
loss 17.45210838317871
w.grad [[-0.36879944801330566], [-1.3713886737823486]]
loss 11.725476264953613
w.grad [[-0.10959792137145996], [-1.0981287956237793]]
loss 8.064153671264648
w.grad [[0.06824100017547607], [-0.9064292907714844]]
loss 5.714572429656982
w.grad [[0.1896425485610962], [-0.7713616490364075]]
loss 4.199885368347168
w.grad [[0.2719000577926636], [-0.6756294369697571]]
loss 3.2

归一化仅对参数空间进行了可逆的线性变换，模型的理论表达能力不变，不改变数据的本质关系。这一现象类似于“换单位不会影响物理规律”。 归一化不会影响深度学习模型的训练结果，因为它只是数据的线性变换，保留了所有必要的信息，模型可以通过权重调整完全补偿这种变换。

因为对feature进行归一化会让训练更稳定，且不会带来任何坏处。基本上所有的深度学习的模型都默认会对feature进行归一化操作。